In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical
from one_round import OneRound
from player import Player

In [2]:
class ValueNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(11, 128)
        self.l2 = nn.Linear(128, 1)

    def forward(self, x):
        x = F.relu(self.l1(x))
        x = self.l2(x)
        return x

In [3]:
class PolicyNet(nn.Module):
    def __init__(self, action_size):
        super().__init__()
        self.l1 = nn.Linear(11,128)
        self.l2 = nn.Linear(128, action_size)

    def forward(self, x):
        x = F.relu(self.l1(x))
        x = F.softmax(self.l2(x), dim=1)
        return x

In [4]:
class Agent:
    def __init__(self):
        self.gamma = 0.98
        self.lr_pi = 0.0002
        self.lr_v = 0.0005
        self.action_size = 3
        # actionは"call",fold","raise"のみとする

        self.pi = PolicyNet(self.action_size)
        self.v = ValueNet()

        self.optimizer_pi = optim.Adam(self.pi.parameters(), lr=self.lr_pi)
        self.optimizer_v = optim.Adam(self.v.parameters(), lr=self.lr_v)

    def get_action(self, state,mask):
        state = torch.tensor(state[np.newaxis, :], dtype=torch.float32)
        mask = torch.tensor(mask,dtype=torch.float32)
        probs = self.pi(state)
        probs = probs[0]
        probs = probs * mask
        m = Categorical(probs)
        action = m.sample().item()
        return action, probs[action]

    def update(self, state, action_prob, reward, next_state, player_done):
        state = torch.tensor(state[np.newaxis, :], dtype=torch.float32)

        if player_done:
            target = torch.tensor(reward, dtype=torch.float32)
        else:
            next_state = torch.tensor(next_state[np.newaxis, :], dtype=torch.float32)
            target = reward + self.gamma * self.v(next_state)

        target.detach()
        v = self.v(state)
        loss_fn = nn.MSELoss()
        loss_v = loss_fn(v, target)

        delta = target - v
        loss_pi = -torch.log(action_prob) * delta.item()
        loss_pi = loss_pi.float()

        self.optimizer_v.zero_grad()
        self.optimizer_pi.zero_grad()
        loss_v.backward()
        loss_pi.backward()
        self.optimizer_v.step()
        self.optimizer_pi.step()
    
    def copy_from(self, other_agent):
        # other_agent からパラメータをコピー
        self.pi.load_state_dict(other_agent.pi.state_dict())
        self.v.load_state_dict(other_agent.v.state_dict())

        self.optimizer_pi.load_state_dict(other_agent.optimizer_pi.state_dict())
        self.optimizer_v.load_state_dict(other_agent.optimizer_v.state_dict())

In [ ]:
agent_a = Agent()
agent_b = Agent()
reward_history = []
epoch_deque = 0
episodes = 1000

for episode in range(episodes):
    print("episode:",episode)
    # ゲームの設定
    # プレイヤーは二人
    players = []
    for i in range(2):
        players.append(Player(10000,"player" + str(i),i))

    one_round = OneRound(players,0,100)
    one_round.set()

    # ゲームの初期条件を入手
    total_reward = 0
    player_index = one_round.current_index
    state_a = one_round.player_state(player_index)

    while True:
        mask_a = one_round.mask(one_round.current_index)
        action_a, prob_a = agent_a.get_action(state_a,mask_a)

        if action_a == 0:
            action_a = "f"
        elif action_a == 1:
            action_a= "c"
        elif action_a == 2:
            action_a = "r"
        
        reward_a, state_b = one_round.step(action_a)
        print(reward_a)

        if action_a == "f":
            agent_a.update(state_a,prob_a,reward_a,None,True)
            break
        if one_round.current_phase == 4:
            agent_a.update(state_a,prob_a,reward_a,None,True)
            break

        mask_b = one_round.mask(one_round.current_index)
        action_b, prob_b = agent_b.get_action(state_b,mask_b)

        if action_b == 0:
            action_b = "f"
        if action_b == 1:
            action_b = "c"
        if action_b == 2:
            action_b = "r"

        reward_b, next_state_a = one_round.step(action_b)

        if action_b == "f":
            reward_a += 0.5
            agent_a.update(state_a,prob_a,reward_a,None,True)
            break
        if one_round.current_phase == 4:
            agent_a.update(state_a,prob_a,reward_a,None,True)
            break

        agent_a.update(state_a,prob_a,reward_a,next_state_a,False)

        state_a = next_state_a
        total_reward += reward_a
    if episode % 20 == 0:
        reward_history.append(total_reward)
        total_reward = 0
    else:
        epoch_deque += total_reward

    if episode % 50 == 0:
        agent_b.copy_from(agent_a)

In [ ]:
import matplotlib.pyplot as plt
# グラフの描画
plt.figure(figsize=(10, 6))  # グラフのサイズを指定
plt.plot(reward_history, label='Reward per Episode', marker='o')

# グラフの装飾
plt.title("Reward History per Episode", fontsize=16)  # タイトル
plt.xlabel("Episode", fontsize=14)  # x軸ラベル
plt.ylabel("Total Reward", fontsize=14)  # y軸ラベル
plt.grid(True)  # グリッド表示